# Stage 0.75: Renewable Energy Profiles (Solar & Wind Integration)

## Dynamic Electric Vehicle Charging Pricing — Renewable Integration (Gap 3)

**Base paper:** Lepolesa et al., "Dynamic Electric Vehicle Charging Pricing for Load Balancing in Power Distribution Networks based on Collaborative DDPG Agents", IEEE Transactions on Smart Grid, 2025.

---

### Gap 3 (from the base paper's Future Work section):
The base paper does not factor renewable generation (solar and wind) into conventional load or dynamic pricing. This notebook derives **real solar and wind generation estimates** directly from the weather measurements already available in `hourly_data.xlsx` (Tetouan, Morocco dataset) and computes the **net load** for each of the $N=4$ distribution networks:

$$\text{Net\_Load}_i(t) = \text{Conventional\_Load}_i(t) - P_{\text{solar}, i}(t) - P_{\text{wind}, i}(t)$$

### Data Sources from `hourly_data.xlsx`:
- `GeneralDiffuseFlows` + `DiffuseFlows`: Real solar diffuse irradiance measurements ($W/m^2$)
- `WindSpeed`: Real wind speed measurements ($m/s$)

### This notebook:
1. Defines physical parameters for rooftop solar PV and micro-wind turbines per network
2. Computes hourly solar PV generation via standard irradiance-to-power models
3. Computes hourly wind generation via standard cubic aerodynamic power curves
4. Calculates net load, net utilization, and renewable-adjusted `util_diff` for all $N=4$ networks
5. Visualizes the renewable impact (midday solar peak, duck curve, net load reduction)
6. Exports `renewable_profiles.csv`, updated `n_network_24h_test.csv`, and `n_network_net_load.csv`

## 1. Imports and Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings

# Reproducibility seed
np.random.seed(42)
warnings.filterwarnings('ignore')

# Plotting aesthetics
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Paths
BASE_REPO_PATH = 'DRLDynamicPricing'
OUTPUT_DIR = 'data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Imports complete.')

## 2. Configuration — Renewable Modeling Parameters

All physical constants, panel areas, and turbine specifications are stated explicitly here.

### Solar Modeling:
$$P_{\text{solar}, i}(t) = I_{\text{total}}(t) \times A_{\text{panel}, i} \times \eta_{\text{panel}}$$
where $I_{\text{total}}(t) = \text{GeneralDiffuseFlows}(t) + \text{DiffuseFlows}(t)$ ($W/m^2$).

| Network | Rooftop Panel Area ($A_{\text{panel}}$) | Panel Efficiency ($\eta$) | Peak Solar Output (approx) |
|---|:---:|:---:|:---:|
| **Residential** | $800\text{ m}^2$ (aggregate neighborhood rooftops) | 18% (monocrystalline Si) | ~240 kW |
| **Commercial** | $600\text{ m}^2$ (rooftop commercial/mall) | 18% (monocrystalline Si) | ~180 kW |
| **Industrial** | $1,800\text{ m}^2$ (large warehouse/factory roofs) | 18% (monocrystalline Si) | ~540 kW |
| **Institutional** | $900\text{ m}^2$ (hospital/university campus buildings) | 18% (monocrystalline Si) | ~270 kW |

### Wind Modeling:
Piecewise aerodynamic power curve:
$$P_{\text{wind}, i}(t) = N_{\text{turbines}, i} \times \begin{cases} 0, & v(t) < v_{\text{cut-in}} \text{ or } v(t) > v_{\text{cut-out}} \\ \frac{1}{2} \rho A_{\text{rotor}} C_p v(t)^3, & v_{\text{cut-in}} \le v(t) < v_{\text{rated}} \\ P_{\text{rated}}, & v_{\text{rated}} \le v(t) \le v_{\text{cut-out}} \end{cases}$$

| Parameter | Value | Justification |
|---|---|---|
| Air Density ($\rho$) | $1.225\text{ kg/m}^3$ | Standard sea-level atmosphere |
| Power Coefficient ($C_p$) | 0.35 | Practical Betz limit fraction for micro-turbines |
| Cut-in Wind Speed ($v_{\text{cut-in}}$) | $1.5\text{ m/s}$ | Low cut-in for urban/distributed wind turbines |
| Rated Wind Speed ($v_{\text{rated}}$) | $9.0\text{ m/s}$ | Standard rated speed for small wind systems |
| Cut-out Wind Speed ($v_{\text{cut-out}}$) | $20.0\text{ m/s}$ | Storm protection cutout |
| Residential Turbines | 5 units, $r=1.8\text{ m}$ ($A=10.18\text{ m}^2$) | Distributed small rooftop wind |
| Commercial Turbines | 10 units, $r=2.0\text{ m}$ ($A=12.57\text{ m}^2$) | Commercial building wind array |
| Industrial Turbines | 30 units, $r=2.5\text{ m}$ ($A=19.63\text{ m}^2$) | Industrial park distributed wind farm |
| Institutional Turbines | 12 units, $r=2.0\text{ m}$ ($A=12.57\text{ m}^2$) | Campus-scale wind generation |

In [ ]:
# ============================================================
# CONFIGURATION — Physical & Environmental Parameters
# ============================================================

NETWORK_NAMES = ['residential', 'commercial', 'industrial', 'institutional']

# Network Maximum Capacities (W) from Stage 0 config
NETWORK_MAX = {
    'residential':   1_166_666,
    'commercial':    1_000_000,
    'industrial':    1_500_000,
    'institutional':   800_000,
}

# Test window parameters
TEST_START = 192 * 3  # Hour 576
TEST_WINDOW = 24

# Solar PV parameters
SOLAR_EFFICIENCY = 0.18  # 18% standard commercial mono-Si efficiency
SOLAR_PANEL_AREAS = {
    'residential':    800,   # m2 (small residential aggregate)
    'commercial':     600,   # m2 (medium commercial rooftop)
    'industrial':    1800,   # m2 (large industrial facility roofs)
    'institutional':  900,   # m2 (medium-large institutional campus)
}

# Wind turbine physical constants
AIR_DENSITY_RHO = 1.225    # kg/m^3 (standard air density)
WIND_CP = 0.35             # Power coefficient
V_CUT_IN = 1.5             # m/s (cut-in speed)
V_RATED = 9.0              # m/s (rated speed)
V_CUT_OUT = 20.0           # m/s (cut-out speed)

# Wind turbine installation configuration per network
WIND_CONFIG = {
    'residential':   {'n_turbines': 5,  'rotor_radius_m': 1.8},
    'commercial':    {'n_turbines': 10, 'rotor_radius_m': 2.0},
    'industrial':    {'n_turbines': 30, 'rotor_radius_m': 2.5},
    'institutional': {'n_turbines': 12, 'rotor_radius_m': 2.0},
}

# Time labels for 24-hour plots
TIME_LABELS = [
    '00:00',' ',' ',' ',' ',' ',' ',' ',' ',' ',' ',' ',
    '12:00',' ',' ',' ',' ',' ',' ',' ',' ',' ',' ','23:00'
]

print('Renewable modeling configuration loaded successfully.')

## 3. Load Weather and Network Load Datasets

We load:
1. `hourly_data.xlsx` for real irradiance (`GeneralDiffuseFlows` + `DiffuseFlows`) and `WindSpeed`
2. `data/n_network_utilization.csv` (generated in Stage 0) for conventional loads of all $N=4$ networks

In [ ]:
# Load real hourly weather data
df_weather = pd.read_excel(os.path.join(BASE_REPO_PATH, 'hourly_data.xlsx'))
N_HOURS = 8736
df_weather = df_weather.iloc[:N_HOURS].reset_index(drop=True)

# Extract solar irradiance and wind speed
gdf = df_weather['GeneralDiffuseFlows'].values
df_flow = df_weather['DiffuseFlows'].values
total_irradiance = gdf + df_flow  # Total solar irradiance in W/m^2
wind_speed = df_weather['WindSpeed'].values  # Wind speed in m/s

# Load Stage 0 conventional load dataset
df_stage0_util = pd.read_csv(os.path.join(OUTPUT_DIR, 'n_network_utilization.csv'))
conv_loads = {
    name: df_stage0_util[f'{name}_load'].values for name in NETWORK_NAMES
}

print(f'Datasets loaded: {N_HOURS} hourly records ({N_HOURS/24:.0f} days)')
print(f'Total Irradiance (W/m2): min={total_irradiance.min():.2f}, max={total_irradiance.max():.2f}, mean={total_irradiance.mean():.2f}')
print(f'Wind Speed (m/s):        min={wind_speed.min():.2f}, max={wind_speed.max():.2f}, mean={wind_speed.mean():.2f}')

## 4. Solar PV Generation Model

Solar power output in Watts for network $i$ at time $t$:

$$P_{\text{solar}, i}(t) = I_{\text{total}}(t) \times A_{\text{panel}, i} \times \eta_{\text{solar}}$$

In [ ]:
def compute_solar_generation(irradiance, panel_area_m2, efficiency=SOLAR_EFFICIENCY):
    """
    Compute solar PV generation in Watts from solar irradiance.
    
    Parameters:
    -----------
    irradiance : np.ndarray
        Hourly total diffuse solar irradiance in W/m^2.
    panel_area_m2 : float
        Total installed rooftop solar panel area in m^2.
    efficiency : float
        Panel efficiency (default: 0.18 for 18%).
        
    Returns:
    --------
    np.ndarray : Solar power generation in Watts.
    """
    return np.maximum(irradiance * panel_area_m2 * efficiency, 0.0)

# Calculate solar generation for each network
solar_generation = {}
for name in NETWORK_NAMES:
    area = SOLAR_PANEL_AREAS[name]
    solar_generation[name] = compute_solar_generation(total_irradiance, area, SOLAR_EFFICIENCY)

print('Solar PV Generation Statistics (Watts):')
print(f'{"Network":<16} {"Area (m2)":>10} {"Min (W)":>10} {"Max (W)":>12} {"Mean (W)":>12} {"Max (kW)":>10}')
print('-' * 76)
for name in NETWORK_NAMES:
    s = solar_generation[name]
    print(f'{name:<16} {SOLAR_PANEL_AREAS[name]:>10} {s.min():>10.0f} {s.max():>12,.0f} {s.mean():>12,.0f} {s.max()/1000:>10.1f}')

## 5. Wind Turbine Generation Model

Wind power generation is computed using the aerodynamic power equation with cut-in, rated, and cut-out thresholds.

In [ ]:
def compute_wind_generation(wind_speeds, n_turbines, rotor_radius_m,
                            v_cut_in=V_CUT_IN, v_rated=V_RATED, v_cut_out=V_CUT_OUT,
                            cp=WIND_CP, rho=AIR_DENSITY_RHO):
    """
    Compute wind power generation in Watts from wind speed.
    
    Parameters:
    -----------
    wind_speeds : np.ndarray
        Hourly wind speed in m/s.
    n_turbines : int
        Number of installed wind turbines.
    rotor_radius_m : float
        Rotor blade radius in meters.
    v_cut_in, v_rated, v_cut_out : float
        Operational cut-in, rated, and cut-out speeds in m/s.
    cp : float
        Turbine aerodynamic power coefficient.
    rho : float
        Air density in kg/m^3.
        
    Returns:
    --------
    np.ndarray : Wind power generation in Watts.
    """
    rotor_area = np.pi * (rotor_radius_m ** 2)
    p_rated_single = 0.5 * rho * rotor_area * cp * (v_rated ** 3)
    
    power_single = np.zeros_like(wind_speeds, dtype=float)
    
    # Region 2: Cubic ramp between cut-in and rated
    mask_ramp = (wind_speeds >= v_cut_in) & (wind_speeds < v_rated)
    power_single[mask_ramp] = 0.5 * rho * rotor_area * cp * (wind_speeds[mask_ramp] ** 3)
    
    # Region 3: Constant rated power between rated and cut-out
    mask_rated = (wind_speeds >= v_rated) & (wind_speeds <= v_cut_out)
    power_single[mask_rated] = p_rated_single
    
    # Region 1 (< cut-in) and Region 4 (> cut-out) remain 0
    return power_single * n_turbines

# Calculate wind generation for each network
wind_generation = {}
for name in NETWORK_NAMES:
    cfg = WIND_CONFIG[name]
    wind_generation[name] = compute_wind_generation(
        wind_speed,
        n_turbines=cfg['n_turbines'],
        rotor_radius_m=cfg['rotor_radius_m']
    )

print('Wind Generation Statistics (Watts):')
print(f'{"Network":<16} {"Turbines":>10} {"Radius (m)":>12} {"Max (W)":>12} {"Mean (W)":>12} {"Max (kW)":>10}')
print('-' * 78)
for name in NETWORK_NAMES:
    w = wind_generation[name]
    cfg = WIND_CONFIG[name]
    print(f'{name:<16} {cfg["n_turbines"]:>10} {cfg["rotor_radius_m"]:>12.1f} {w.max():>12,.0f} {w.mean():>12,.0f} {w.max()/1000:>10.1f}')

## 6. Combine into Net Load and Compute Net Utilization

$$\text{Net\_Load}_i(t) = \max\left(\text{Conventional\_Load}_i(t) - P_{\text{solar}, i}(t) - P_{\text{wind}, i}(t), \; 0.05 \times \text{Conventional\_Load}_i(t)\right)$$

Clipping at 5% of conventional load prevents unrealistic negative grid reverse flows while allowing renewables to reduce load significantly by up to 95% during peak solar/wind hours.

In [ ]:
net_loads = {}
net_utilizations = {}
total_renewables = {}

for name in NETWORK_NAMES:
    tot_ren = solar_generation[name] + wind_generation[name]
    total_renewables[name] = tot_ren
    
    # Compute net load with defensive lower bound
    raw_net = conv_loads[name] - tot_ren
    net_loads[name] = np.maximum(raw_net, 0.05 * conv_loads[name])
    
    # Compute net utilization (net load / max capacity)
    net_utilizations[name] = net_loads[name] / NETWORK_MAX[name]

# Compute generalized N-network util_diff on net utilization
def compute_n_network_util_diff(util_dict):
    names = list(util_dict.keys())
    diffs = {}
    for i, name_i in enumerate(names):
        other_utils = [util_dict[name_j] for j, name_j in enumerate(names) if j != i]
        mean_other = np.mean(other_utils, axis=0)
        diffs[name_i] = np.clip(util_dict[name_i] - mean_other, 0, None)
    return diffs

net_util_diffs = compute_n_network_util_diff(net_utilizations)

print('Conventional vs. Net Load Comparison (Full 8736 Hours):')
print(f'{"Network":<16} {"Conv Mean (kW)":>15} {"Renewable Mean (kW)":>20} {"Net Mean (kW)":>15} {"Load Reduction":>16}')
print('-' * 85)
for name in NETWORK_NAMES:
    c_m = conv_loads[name].mean() / 1000
    r_m = total_renewables[name].mean() / 1000
    n_m = net_loads[name].mean() / 1000
    reduction = (1 - n_m / c_m) * 100
    print(f'{name:<16} {c_m:>15.1f} {r_m:>20.1f} {n_m:>15.1f} {reduction:>15.1f}%')

## 7. Extract 24-Hour Test Window

Extract hours 576 to 599 ($TEST\_START = 192 \times 3$) matching the base paper's evaluation benchmark.

In [ ]:
test_solar = {name: solar_generation[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}
test_wind = {name: wind_generation[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}
test_ren = {name: total_renewables[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}
test_conv = {name: conv_loads[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}
test_net = {name: net_loads[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}
test_net_util = {name: net_utilizations[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}
test_net_diff = {name: net_util_diffs[name][TEST_START:TEST_START+TEST_WINDOW] for name in NETWORK_NAMES}

print(f'Test Window (Hour {TEST_START} to {TEST_START+TEST_WINDOW-1}) Renewable Impact:')
print(f'{"Network":<16} {"Conv Peak (kW)":>15} {"Solar Peak (kW)":>16} {"Wind Peak (kW)":>15} {"Net Min (kW)":>14} {"Midday Reduction":>18}')
print('-' * 98)
for name in NETWORK_NAMES:
    cp = test_conv[name].max() / 1000
    sp = test_solar[name].max() / 1000
    wp = test_wind[name].max() / 1000
    nm = test_net[name].min() / 1000
    midday_red = ((test_conv[name][12:15] - test_net[name][12:15]) / test_conv[name][12:15]).mean() * 100
    print(f'{name:<16} {cp:>15.1f} {sp:>16.1f} {wp:>15.1f} {nm:>14.1f} {midday_red:>17.1f}%')

## 8. Visualizations

### Plot 1: Solar & Wind Generation Profiles (Test Window)

In [ ]:
hours = np.arange(24)
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Solar generation
for idx, name in enumerate(NETWORK_NAMES):
    ax1.plot(hours, test_solar[name] / 1000, color=colors[idx], linewidth=2.5, marker='o', markersize=4, label=name.capitalize())
    ax1.fill_between(hours, test_solar[name] / 1000, color=colors[idx], alpha=0.15)

ax1.set_title('Solar PV Generation (24-Hour Test Window)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Hour of Day', fontsize=12)
ax1.set_ylabel('Generation (kW)', fontsize=12)
ax1.set_xticks(hours)
ax1.set_xticklabels(TIME_LABELS, fontsize=9)
ax1.legend(fontsize=10)
ax1.set_xlim(0, 23)

# Wind generation
for idx, name in enumerate(NETWORK_NAMES):
    ax2.plot(hours, test_wind[name] / 1000, color=colors[idx], linewidth=2.5, marker='s', markersize=4, label=name.capitalize())
    ax2.fill_between(hours, test_wind[name] / 1000, color=colors[idx], alpha=0.15)

ax2.set_title('Wind Power Generation (24-Hour Test Window)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Hour of Day', fontsize=12)
ax2.set_ylabel('Generation (kW)', fontsize=12)
ax2.set_xticks(hours)
ax2.set_xticklabels(TIME_LABELS, fontsize=9)
ax2.legend(fontsize=10)
ax2.set_xlim(0, 23)

plt.tight_layout()
plt.show()

### Plot 2: Conventional vs. Net Load (All 4 Networks)

Demonstrates the **duck curve effect** where midday solar generation depresses grid net demand.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Conventional Load vs. Net Load — 24-Hour Test Window (All 4 Networks)', fontsize=16, fontweight='bold')

for idx, (name, color) in enumerate(zip(NETWORK_NAMES, colors)):
    ax = axes[idx // 2, idx % 2]
    conv_kw = test_conv[name] / 1000
    net_kw = test_net[name] / 1000
    ren_kw = test_ren[name] / 1000
    
    ax.plot(hours, conv_kw, color='grey', linestyle='--', linewidth=2, label='Conventional Load')
    ax.plot(hours, net_kw, color=color, linewidth=2.5, marker='o', markersize=4, label='Net Load (with Renewables)')
    ax.fill_between(hours, net_kw, conv_kw, color='gold', alpha=0.3, hatch='//', label='Renewable Generation')
    
    ax.set_title(f'{name.capitalize()} Network', fontsize=13, fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Load (kW)')
    ax.set_xticks(hours[::3])
    ax.set_xticklabels([f'{h:02d}:00' for h in hours[::3]], rotation=45)
    ax.legend(fontsize=9, loc='upper left')
    ax.set_xlim(0, 23)

plt.tight_layout()
plt.show()

### Plot 3: Net Load Overlay & Net Utilization vs. Conventional Utilization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Net Load Overlay
for idx, name in enumerate(NETWORK_NAMES):
    ax1.plot(hours, test_net[name] / 1000, color=colors[idx], linewidth=2.5, marker='o', markersize=4, label=name.capitalize())

ax1.set_title('Net Load Profiles (All Networks Overlay)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Hour of Day', fontsize=12)
ax1.set_ylabel('Net Load (kW)', fontsize=12)
ax1.set_xticks(hours)
ax1.set_xticklabels(TIME_LABELS, fontsize=9)
ax1.legend(fontsize=10)
ax1.set_xlim(0, 23)

# Net Utilization
for idx, name in enumerate(NETWORK_NAMES):
    ax2.plot(hours, test_net_util[name], color=colors[idx], linewidth=2.5, marker='s', markersize=4, label=name.capitalize())

ax2.set_title('Net Utilization (Net Load / Capacity)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Hour of Day', fontsize=12)
ax2.set_ylabel('Net Utilization', fontsize=12)
ax2.set_xticks(hours)
ax2.set_xticklabels(TIME_LABELS, fontsize=9)
ax2.legend(fontsize=10)
ax2.set_xlim(0, 23)

plt.tight_layout()
plt.show()

## 9. Export Updated Datasets

We export:
1. `data/renewable_profiles.csv`: Full 8736-hour solar, wind, and total renewable series for all 4 networks
2. `data/n_network_24h_test.csv` (updated): 24-hour test window with conventional load, net load, net utilization, and net util_diff
3. `data/n_network_net_load.csv`: Full 8736-hour dataset with all conventional, renewable, and net columns
4. `data/renewable_config.csv`: Configuration parameters for solar/wind modeling

In [ ]:
# 1. Export renewable_profiles.csv
df_ren_export = pd.DataFrame({'Hour_index': range(N_HOURS)})
for name in NETWORK_NAMES:
    df_ren_export[f'{name}_solar_W'] = solar_generation[name]
    df_ren_export[f'{name}_wind_W'] = wind_generation[name]
    df_ren_export[f'{name}_total_renewable_W'] = total_renewables[name]

ren_csv_path = os.path.join(OUTPUT_DIR, 'renewable_profiles.csv')
df_ren_export.to_csv(ren_csv_path, index=False)
print(f'1. Written: {ren_csv_path} ({len(df_ren_export)} rows x {len(df_ren_export.columns)} cols)')

# 2. Update n_network_24h_test.csv with net-load columns
test_csv_path = os.path.join(OUTPUT_DIR, 'n_network_24h_test.csv')
df_test_existing = pd.read_csv(test_csv_path)

for name in NETWORK_NAMES:
    df_test_existing[f'{name}_solar_W'] = test_solar[name]
    df_test_existing[f'{name}_wind_W'] = test_wind[name]
    df_test_existing[f'{name}_net_load'] = test_net[name]
    df_test_existing[f'{name}_net_utilization'] = test_net_util[name]
    df_test_existing[f'{name}_net_util_diff'] = test_net_diff[name]

df_test_existing.to_csv(test_csv_path, index=False)
print(f'2. Updated: {test_csv_path} ({len(df_test_existing)} rows x {len(df_test_existing.columns)} cols)')

# 3. Export n_network_net_load.csv (Full 8736 hours)
df_net_full = pd.DataFrame({'Hour_index': range(N_HOURS)})
for name in NETWORK_NAMES:
    df_net_full[f'{name}_conv_load'] = conv_loads[name]
    df_net_full[f'{name}_solar_W'] = solar_generation[name]
    df_net_full[f'{name}_wind_W'] = wind_generation[name]
    df_net_full[f'{name}_net_load'] = net_loads[name]
    df_net_full[f'{name}_net_utilization'] = net_utilizations[name]
    df_net_full[f'{name}_net_util_diff'] = net_util_diffs[name]

net_full_path = os.path.join(OUTPUT_DIR, 'n_network_net_load.csv')
df_net_full.to_csv(net_full_path, index=False)
print(f'3. Written: {net_full_path} ({len(df_net_full)} rows x {len(df_net_full.columns)} cols)')

# 4. Export renewable_config.csv
ren_config_data = []
for name in NETWORK_NAMES:
    ren_config_data.append({
        'network_name': name,
        'panel_area_m2': SOLAR_PANEL_AREAS[name],
        'panel_efficiency': SOLAR_EFFICIENCY,
        'n_turbines': WIND_CONFIG[name]['n_turbines'],
        'rotor_radius_m': WIND_CONFIG[name]['rotor_radius_m'],
        'air_density_kg_m3': AIR_DENSITY_RHO,
        'wind_cp': WIND_CP,
        'v_cut_in_mps': V_CUT_IN,
        'v_rated_mps': V_RATED,
        'v_cut_out_mps': V_CUT_OUT,
    })
df_ren_config = pd.DataFrame(ren_config_data)
ren_config_path = os.path.join(OUTPUT_DIR, 'renewable_config.csv')
df_ren_config.to_csv(ren_config_path, index=False)
print(f'4. Written: {ren_config_path}')
print()
print(df_ren_config.to_string(index=False))

## 10. Verification of Exported Files

In [ ]:
print('='*60)
print('RENEWABLE PROFILES EXPORT VERIFICATION')
print('='*60)

verification_list = [
    ('renewable_profiles.csv', N_HOURS, 1 + 3 * len(NETWORK_NAMES)),
    ('n_network_24h_test.csv', 24, 2 + 3 * len(NETWORK_NAMES) + 5 * len(NETWORK_NAMES)),
    ('n_network_net_load.csv', N_HOURS, 1 + 6 * len(NETWORK_NAMES)),
    ('renewable_config.csv', len(NETWORK_NAMES), 10),
]

all_ok = True
for fname, exp_r, exp_c in verification_list:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if not os.path.exists(fpath):
        print(f'  [MISSING] {fname}')
        all_ok = False
        continue
    df_c = pd.read_csv(fpath)
    r_ok = len(df_c) == exp_r
    c_ok = len(df_c.columns) == exp_c
    status = 'OK' if (r_ok and c_ok) else 'MISMATCH'
    if not (r_ok and c_ok):
        all_ok = False
    print(f'  [{status}] {fname}: {len(df_c)} rows x {len(df_c.columns)} cols (expected {exp_r} x {exp_c})')

print()
if all_ok:
    print('All renewable datasets exported and verified successfully.')
else:
    print('WARNING: Shape mismatches encountered.')

## 11. Summary — What to Report in the Paper

In [ ]:
print('='*70)
print('STAGE 0.75 SUMMARY -- WHAT TO REPORT IN THE PAPER')
print('='*70)
print()
print('1. RENEWABLE ENERGY INTEGRATION (Gap 3):')
print('   Extended base paper pricing model by subtracting localized solar PV')
print('   and wind generation estimates from conventional loads before pricing.')
print('   Derived from real GeneralDiffuseFlows, DiffuseFlows, and WindSpeed')
print('   measurements from the Tetouan, Morocco dataset in hourly_data.xlsx.')
print()
print('2. PHYSICAL MODELING SPECIFICATIONS:')
print(f'   Solar Efficiency: {SOLAR_EFFICIENCY*100:.0f}% (Monocrystalline Si)')
for name in NETWORK_NAMES:
    print(f'   - {name.capitalize():<14}: {SOLAR_PANEL_AREAS[name]:>4} m2 rooftop PV | {WIND_CONFIG[name]["n_turbines"]:>2} wind turbines (r={WIND_CONFIG[name]["rotor_radius_m"]}m)')
print(f'   Wind Turbine Model: v_cut_in={V_CUT_IN} m/s, v_rated={V_RATED} m/s, v_cut_out={V_CUT_OUT} m/s, Cp={WIND_CP}')
print()
print('3. RENEWABLE IMPACT ON TEST WINDOW (24 Hours):')
print(f'{"Network":<16} {"Conv Mean (kW)":>15} {"Net Mean (kW)":>15} {"Midday Solar Drop":>20}')
print('-' * 70)
for name in NETWORK_NAMES:
    c_mean = test_conv[name].mean() / 1000
    n_mean = test_net[name].mean() / 1000
    midday_drop = ((test_conv[name][12:15] - test_net[name][12:15]) / test_conv[name][12:15]).mean() * 100
    print(f'{name:<16} {c_mean:>15.1f} {n_mean:>15.1f} {midday_drop:>19.1f}%')
print()
print('4. EXPORTED FILES:')
print(f'   - {os.path.join(OUTPUT_DIR, "renewable_profiles.csv")}')
print(f'   - {os.path.join(OUTPUT_DIR, "n_network_24h_test.csv")}')
print(f'   - {os.path.join(OUTPUT_DIR, "n_network_net_load.csv")}')
print(f'   - {os.path.join(OUTPUT_DIR, "renewable_config.csv")}')